In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import pandas as pd
import sqlite3

In [3]:
train = pd.read_csv('../data/train_cleaned.csv', parse_dates=['Date'])

In [4]:
conn = sqlite3.connect('../data/rossmann.db')
train.to_sql('sales', conn, if_exists='replace', index=False)


844392

In [6]:
result = pd.read_sql("SELECT COUNT(*) as total_rows FROM sales", conn)
print("DB created successfully!")
print(f"Total rows loaded: {result['total_rows'][0]:,}")

DB created successfully!
Total rows loaded: 844,392


In [7]:
def run(query):
    return pd.read_sql(query, conn)

In [8]:
# Query 1: Total & Average Sales by Store Type
q1 = """
SELECT 
    StoreType,
    COUNT(*) as total_days,
    COUNT(DISTINCT Store) as num_stores,
    ROUND(SUM(Sales), 0) as total_sales,
    ROUND(AVG(Sales), 0) as avg_daily_sales,
    ROUND(SUM(Sales) / COUNT(DISTINCT Store), 0) as avg_sales_per_store
FROM sales
GROUP BY StoreType
ORDER BY avg_daily_sales DESC
"""

run(q1)

,StoreType,total_days,num_stores,total_sales,avg_daily_sales,avg_sales_per_store
0,b,15563,17,1.592314e+08,10231.0,9366552.0
1,c,112978,148,7.832214e+08,6933.0,5292036.0
2,a,457077,602,3.165335e+09,6925.0,5258031.0
3,d,258774,348,1.765393e+09,6822.0,5072968.0


In [9]:
# Query 2: Promo vs Non-Promo Sales
q2 = """
SELECT 
    Promo,
    COUNT(*) as total_days,
    ROUND(AVG(Sales), 0) as avg_sales,
    ROUND(AVG(Customers), 0) as avg_customers,
    ROUND(AVG(Sales) * 1.0 / AVG(Customers), 2) as sales_per_customer
FROM sales
GROUP BY Promo
"""

run(q2)

,Promo,total_days,avg_sales,avg_customers,sales_per_customer
0,0,467496,5929.0,697.0,8.51
1,1,376896,8228.0,844.0,9.74


In [10]:
# Query 3: Top 10 Stores by Avg Daily Sales
q3 = """
SELECT 
    Store,
    StoreType,
    Assortment,
    ROUND(AVG(Sales), 0) as avg_daily_sales,
    ROUND(AVG(Customers), 0) as avg_customers,
    COUNT(*) as days_open
FROM sales
GROUP BY Store
ORDER BY avg_daily_sales DESC
LIMIT 10
"""

run(q3)

,Store,StoreType,Assortment,avg_daily_sales,avg_customers,days_open
0,817,a,a,21757.0,3131.0,784
1,262,b,a,20719.0,3402.0,942
2,1114,a,c,20667.0,3201.0,784
3,251,a,c,19123.0,2450.0,779
4,842,d,c,18575.0,1149.0,622
5,513,a,a,18179.0,2096.0,784
6,562,b,c,17970.0,3105.0,942
7,788,a,c,17962.0,1718.0,784
8,383,a,c,17295.0,2205.0,780
9,756,a,c,16575.0,2347.0,779


In [11]:
# Query 4: Monthly Sales Trend
q4 = """
SELECT 
    strftime('%Y', Date) as year,
    strftime('%m', Date) as month,
    ROUND(AVG(Sales), 0) as avg_sales,
    ROUND(SUM(Sales), 0) as total_sales,
    COUNT(DISTINCT Store) as active_stores
FROM sales
GROUP BY year, month
ORDER BY year, month
"""

run(q4)

,year,month,avg_sales,total_sales,active_stores
0,2013,01,6240.0,180132207.0,1112
1,2013,02,6429.0,171534275.0,1112
2,2013,03,7213.0,201180369.0,1112
3,2013,04,6579.0,183431432.0,1113
4,2013,05,7076.0,185411063.0,1113
5,2013,06,6467.0,180702351.0,1113
6,2013,07,6923.0,208843882.0,1115
7,2013,08,6596.0,198042727.0,1115
8,2013,09,6363.0,178053963.0,1115
9,2013,10,6473.0,187662330.0,1115


In [12]:
# Query 5: Holiday Impact on Sales
q5 = """
SELECT 
    CASE 
        WHEN StateHoliday = '0' AND SchoolHoliday = 0 THEN 'Normal Day'
        WHEN StateHoliday != '0' AND SchoolHoliday = 0 THEN 'State Holiday Only'
        WHEN StateHoliday = '0' AND SchoolHoliday = 1 THEN 'School Holiday Only'
        WHEN StateHoliday != '0' AND SchoolHoliday = 1 THEN 'Both Holidays'
    END as day_type,
    COUNT(*) as total_days,
    ROUND(AVG(Sales), 0) as avg_sales,
    ROUND(AVG(Customers), 0) as avg_customers
FROM sales
GROUP BY day_type
ORDER BY avg_sales DESC
"""

run(q5)

,day_type,total_days,avg_sales,avg_customers
0,Both Holidays,314,8907.0,1485.0
1,State Holiday Only,596,8757.0,1304.0
2,School Holiday Only,163143,7197.0,782.0
3,Normal Day,680339,6895.0,757.0


In [13]:
# Query 6: Competition Distance vs Sales
q6 = """
SELECT 
    CASE 
        WHEN CompetitionDistance < 500 THEN 'Very Close (<500m)'
        WHEN CompetitionDistance < 1000 THEN 'Close (500-1km)'
        WHEN CompetitionDistance < 5000 THEN 'Medium (1-5km)'
        WHEN CompetitionDistance < 10000 THEN 'Far (5-10km)'
        ELSE 'Very Far (>10km)'
    END as competition_bucket,
    COUNT(DISTINCT Store) as num_stores,
    ROUND(AVG(Sales), 0) as avg_sales,
    ROUND(AVG(Customers), 0) as avg_customers
FROM sales
GROUP BY competition_bucket
ORDER BY avg_sales DESC
"""

run(q6)

,competition_bucket,num_stores,avg_sales,avg_customers
0,Very Close (<500m),218,7614.0,969.0
1,Very Far (>10km),187,6825.0,660.0
2,Medium (1-5km),437,6820.0,733.0
3,Far (5-10km),163,6752.0,632.0
4,Close (500-1km),110,6697.0,838.0


In [14]:
# Query 7: Promo2 (Continuous Promo) Effect
q7 = """
SELECT 
    Promo2,
    ROUND(AVG(Sales), 0) as avg_sales,
    ROUND(AVG(Customers), 0) as avg_customers,
    COUNT(DISTINCT Store) as num_stores,
    COUNT(*) as total_days
FROM sales
GROUP BY Promo2
"""

run(q7)

,Promo2,avg_sales,avg_customers,num_stores,total_days
0,0,7351.0,844.0,544,423307
1,1,6558.0,681.0,571,421085


In [15]:
# Query 8: Best Performing Day x Store Type Combo
q8 = """
SELECT 
    StoreType,
    DayOfWeek,
    ROUND(AVG(Sales), 0) as avg_sales,
    ROUND(AVG(Customers), 0) as avg_customers,
    COUNT(*) as total_days
FROM sales
GROUP BY StoreType, DayOfWeek
ORDER BY avg_sales DESC
LIMIT 15
"""

run(q8)

,StoreType,DayOfWeek,avg_sales,avg_customers,total_days
0,b,7,11450.0,2091.0,2216
1,b,1,11055.0,2122.0,2215
2,b,5,10709.0,2178.0,2231
3,b,4,10208.0,2042.0,2230
4,b,2,10172.0,2012.0,2228
5,b,3,10014.0,1998.0,2228
6,a,1,8337.0,910.0,74603
7,b,6,8011.0,1709.0,2215
8,c,1,7989.0,893.0,18458
9,d,1,7954.0,676.0,42284


In [16]:
# Query 9: Stores with Highest Promo Lift %
q9 = """
SELECT 
    Store,
    StoreType,
    ROUND(AVG(CASE WHEN Promo = 1 THEN Sales END), 0) as avg_promo_sales,
    ROUND(AVG(CASE WHEN Promo = 0 THEN Sales END), 0) as avg_normal_sales,
    ROUND(
        (AVG(CASE WHEN Promo = 1 THEN Sales END) - AVG(CASE WHEN Promo = 0 THEN Sales END)) 
        * 100.0 / AVG(CASE WHEN Promo = 0 THEN Sales END), 1
    ) as promo_lift_pct
FROM sales
GROUP BY Store
HAVING avg_promo_sales IS NOT NULL AND avg_normal_sales IS NOT NULL
ORDER BY promo_lift_pct DESC
LIMIT 10
"""

run(q9)

,Store,StoreType,avg_promo_sales,avg_normal_sales,promo_lift_pct
0,198,a,4295.0,1753.0,144.9
1,607,a,5543.0,2552.0,117.2
2,543,c,3926.0,1868.0,110.2
3,575,a,7586.0,3614.0,109.9
4,693,d,9108.0,4578.0,99.0
5,271,a,10746.0,5453.0,97.1
6,552,a,10566.0,5412.0,95.2
7,96,a,7048.0,3638.0,93.7
8,635,a,8048.0,4159.0,93.5
9,489,a,9950.0,5143.0,93.5


In [17]:
# Query 10: Assortment Type vs Sales
q10 = """
SELECT 
    Assortment,
    COUNT(DISTINCT Store) as num_stores,
    ROUND(AVG(Sales), 0) as avg_daily_sales,
    ROUND(AVG(Customers), 0) as avg_customers,
    ROUND(AVG(Sales) * 1.0 / AVG(Customers), 2) as sales_per_customer,
    ROUND(MAX(Sales), 0) as max_single_day_sales
FROM sales
GROUP BY Assortment
ORDER BY avg_daily_sales DESC
"""

run(q10)

,Assortment,num_stores,avg_daily_sales,avg_customers,sales_per_customer,max_single_day_sales
0,b,9,8639.0,2067.0,4.18,22137.0
1,c,513,7301.0,752.0,9.71,41551.0
2,a,593,6621.0,748.0,8.85,38722.0
